# Learned predictive JEPA representations for persistent parts

This notebook reproduces the frozen Day 7 experiment: a new recurrent part head and future-latent predictor trained over **frozen V-JEPA 2.1** features, with no SAM model. The decisive comparison is predictive supervision versus the identical retrieval-only architecture, paired across three predetermined training seeds. A coordinate-only control diagnoses memorization of the synthetic motion schedules.

Select **Runtime → Change runtime type → T4 GPU**, then run through training. **Do not run held-out extraction until the generated checkpoint freeze has been reviewed and committed.** The held-out cell defaults OFF and verifies the exact committed JSON bytes before extraction. A separate integrity gate validates all nine trained checkpoints, complete 30-epoch curves, development selections, source, projection, centering and feature-cache provenance.

This is a supervised part-label experiment in a procedural 2-D renderer, not self-supervised foundation training or automatic part discovery. The head uses current/past feature steps, but the frozen backbone sees offline 16-frame windows. No real-video, face, 3-D or generative-consistency claim follows.

This repository notebook has **empty outputs** and has not been separately executed end to end in a fresh GPU runtime. It follows the command sequence from the research Colab and adds the pinned, audited post-test analysis, mechanism diagnostics and artifact exporter. Training is ongoing when this notebook revision is prepared; the presence of a command here does not establish that its results exist. Saved run artifacts are the evidence for execution and measured results.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, shutil, subprocess, sys, zipfile

SOURCE_REV = "9af4259467caa0bdf7980ef83d856fae1f70f61f"
REPO_URL = "https://github.com/skbdps/vjepa-poc.git"
REPO = Path("/content/vjepa-poc-day7")
UPSTREAM_URL = "https://github.com/facebookresearch/vjepa2.git"
UPSTREAM_REV = "204698b45b3712590f06245fbfba32d3be539812"
UPSTREAM = Path("/content/vjepa2-day7")
FEATURES = Path("/content/day7_features")
RUN = Path("/content/day7_learned_run")
EXP = REPO / "experiments/day7"

# Thread limits apply only to child processes; do not mutate this notebook's environment.
CHILD_ENV = dict(os.environ, OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")

def checkout_exact(url, folder, revision):
    if not (folder / ".git").exists():
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", url, str(folder)], check=True, env=CHILD_ENV)
    subprocess.run(["git", "-C", str(folder), "fetch", "origin", revision], check=True, env=CHILD_ENV)
    subprocess.run(["git", "-C", str(folder), "checkout", "--detach", revision], check=True, env=CHILD_ENV)
    actual = subprocess.check_output(["git", "-C", str(folder), "rev-parse", "HEAD"], text=True).strip()
    assert actual == revision

checkout_exact(REPO_URL, REPO, SOURCE_REV)
checkout_exact(UPSTREAM_URL, UPSTREAM, UPSTREAM_REV)
FEATURES.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print("Experiment source:", SOURCE_REV)
print("Official backbone source:", UPSTREAM_REV)

## Match the research setup without replacing Colab's PyTorch stack

The research runtime used PyTorch `2.11.0+cu130`, `timm==1.0.26`, NumPy, Pillow, OpenCV and einops. Keep the matching PyTorch/torchvision installation provided by the selected Colab runtime. Record its actual versions below rather than claiming all future Colab images will be identical. The official checkpoint download can be several gigabytes; backbone weights are not copied into run archives.

In [ ]:
from importlib.util import find_spec
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.26", "einops", "pillow"], check=True, env=CHILD_ENV)
if find_spec("cv2") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless==4.11.0.86"], check=True, env=CHILD_ENV)
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True, env=CHILD_ENV)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True, env=CHILD_ENV)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before proceeding."
import numpy as np
metadata = {"source_revision": SOURCE_REV, "upstream_revision": UPSTREAM_REV,
            "started_utc": datetime.now(timezone.utc).isoformat(),
            "python": platform.python_version(), "torch": torch.__version__,
            "numpy": np.__version__, "gpu": torch.cuda.get_device_name(),
            "child_thread_limits": {"OMP_NUM_THREADS": 2, "MKL_NUM_THREADS": 2},
            "notebook_status": "reproduction recipe; this notebook was not separately clean-runtime GPU-executed"}
(RUN / "notebook_manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
print(json.dumps(metadata, indent=2))

def run_logged(command, name):
    print("Running:", " ".join(map(str, command)), flush=True)
    with (RUN / f"{name}.log").open("a") as log:
        process = subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1, env=CHILD_ENV)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

for module in ("data.py", "model.py", "train.py", "analyze.py", "diagnose.py", "export_artifacts.py"):
    command = [sys.executable, "-u", EXP / module] + ([] if module == "model.py" else ["--self-test"])
    run_logged(command, module.removesuffix(".py") + "_self_test")

## Extract only training and development features

This opens 36 training and 9 development clips. Test seeds are not rendered or encoded. The extractor uses the pinned official `vjepa2_1_vit_large_384`, four disjoint 16-frame windows, a fixed orthogonal 1024→256 projection, and training-only centering. It also caches full-feature matching and the fixed Day 4 context/motion baseline predictions.

The centering mean, projection and each cache are checksummed. Exact completed caches can be reused; mismatched source or input hashes fail rather than being silently accepted.

In [ ]:
run_logged([sys.executable, "-u", EXP / "extract.py", "--out", FEATURES,
            "--upstream", UPSTREAM, "--splits", "train", "dev"], "train_dev_extraction")
feature_manifest = json.loads((FEATURES / "manifest.json").read_text())
assert sum(c["split"] == "train" for c in feature_manifest["clips"]) == 36
assert sum(c["split"] == "dev" for c in feature_manifest["clips"]) == 9
assert not any(c["split"] == "test" for c in feature_manifest["clips"]), "Use a fresh feature directory for a fresh blind run."
assert feature_manifest["mean"]["fit_split"] == "train"
print("Train/dev caches complete. Test features have not been opened.")

## Train all nine fixed head configurations

The arms are `retrieval_only`, `predictive`, and `coordinate_only`, each trained with seeds 1701, 1702 and 1703 for 30 epochs, batch size 2, AdamW learning rate 3e-4 and weight decay 1e-4. Initial parameters and scene orders are paired across arms. The predictive arm alone adds the horizon-8 future cosine/contrastive objective. Future teacher targets are detached.

Development selects the earliest epoch with maximum mean of visible localization and absent specificity. Learned presence uses a fixed factorized visibility cutoff of 0.5. Baseline thresholds are selected on development only. All seeds are retained; there is no best-seed selection.

If batch size 2 runs out of memory, stop and document a **global** batch-size-1 fallback in a fresh output directory for all arms. Do not quietly change the budget for only one arm. The default command below uses the fixed batch size 2.

In [ ]:
run_logged([sys.executable, "-u", EXP / "train.py", "--stage", "train",
            "--features", FEATURES, "--out", RUN, "--batch-scenes", "2"], "head_training")
FREEZE = RUN / "checkpoint_freeze.json"
freeze = json.loads(FREEZE.read_text())
assert len(freeze["models"]) == 9 and freeze["test_accessed"] is False
summary = json.loads((RUN / "training_summary.json").read_text())
for selected in summary["models"]:
    print(selected["arm"], selected["seed"], "selected epoch", selected["epoch"],
          "best dev utility", round(selected["development"]["utility"], 6),
          "final dev utility", round(selected["last_epoch_development"]["utility"], 6))
freeze_sha = hashlib.sha256(FREEZE.read_bytes()).hexdigest()
print("CHECKPOINT FREEZE:", FREEZE)
print("SHA-256:", freeze_sha)
print("STOP: review and commit this exact JSON before enabling the held-out cell.")

## Stop here: preserve and commit the checkpoint freeze

Review `checkpoint_freeze.json`, the nine development selections and complete curves. Save the trained-head checkpoints before the temporary runtime disappears. Commit the **exact generated freeze JSON** to your GitHub repository and record its immutable commit SHA and repository-relative path. The freeze binds selected weights, all training seeds, source, centering/projection and train/dev caches; it must precede access to the held-out clips.

This notebook does not push to GitHub or claim a freeze has been committed. The next cell defaults OFF. Merely viewing or downloading the JSON is not a commit. Do not use test scores to choose a seed, epoch, threshold or new policy.

The small review archive below contains trained head weights and training evidence, not the large frozen backbone or feature caches. The final exporter preserves the full original predictions and heads in one archive, and emits a separate compact Git evidence tree. An optional feature-cache supplement can be downloaded separately.

In [ ]:
from google.colab import files
review_archive = Path("/content/Day7_Training_Freeze_Review.zip")
with zipfile.ZipFile(review_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and "test" not in path.relative_to(RUN).parts:
            bundle.write(path, "run/" + str(path.relative_to(RUN)))
    for name in ("manifest.json", "mean.npy", "projection.npy"):
        bundle.write(FEATURES / name, "features/" + name)
print("Freeze JSON:", FREEZE, freeze_sha)
print("Review archive:", review_archive, f"{review_archive.stat().st_size / 2**20:.1f} MiB")
files.download(str(FREEZE))
files.download(str(review_archive))

## Held-out stage: enable only after the exact freeze is committed

Fill in the commit SHA and path of your saved freeze, then set `RUN_HELD_OUT = True`. The code fetches that immutable commit and compares its file bytes with the local freeze. It does not check out a different experiment implementation. The extractor then performs its stronger nine-checkpoint/curve/provenance validation before opening any test scene.

The complete stage runs all 18 test clips and reports every predetermined seed, then independently audits the saved predictions and runs fixed-checkpoint mechanism diagnostics for all six visual heads. The diagnostics measure reliance and do not train or select a new method. Do not stop because an intermediate result is unfavorable. This cell is skipped by default, including when using Run all.

In [ ]:
RUN_HELD_OUT = False
FROZEN_CHECKPOINT_COMMIT = ""  # Full40-character Git commit SHA containing the exact freeze JSON.
FROZEN_CHECKPOINT_PATH = "experiments/day7/run_2026-10-09/checkpoint_freeze.json"  # Replace for your committed reproduction freeze.

if RUN_HELD_OUT:
    import re
    assert re.fullmatch(r"[0-9a-f]{40}", FROZEN_CHECKPOINT_COMMIT), "Commit the freeze and supply its full SHA first."
    assert FROZEN_CHECKPOINT_PATH and not Path(FROZEN_CHECKPOINT_PATH).is_absolute()
    assert ".." not in Path(FROZEN_CHECKPOINT_PATH).parts
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", FROZEN_CHECKPOINT_COMMIT], check=True, env=CHILD_ENV)
    committed_freeze = subprocess.check_output(["git", "-C", str(REPO), "show",
                                               f"{FROZEN_CHECKPOINT_COMMIT}:{FROZEN_CHECKPOINT_PATH}"])
    assert committed_freeze == FREEZE.read_bytes(), "Committed freeze bytes differ from the selected local checkpoints."
    metadata["checkpoint_freeze_commit"] = FROZEN_CHECKPOINT_COMMIT
    metadata["checkpoint_freeze_repository_path"] = FROZEN_CHECKPOINT_PATH
    metadata["checkpoint_freeze_sha256"] = hashlib.sha256(committed_freeze).hexdigest()
    (RUN / "notebook_manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
    run_logged([sys.executable, "-u", EXP / "extract.py", "--out", FEATURES,
                "--upstream", UPSTREAM, "--splits", "test", "--checkpoint-freeze", FREEZE], "held_out_extraction")
    run_logged([sys.executable, "-u", EXP / "train.py", "--stage", "test",
                "--features", FEATURES, "--out", RUN, "--frozen-path", FREEZE], "held_out_evaluation")
    run_logged([sys.executable, "-u", EXP / "analyze.py", "--run", RUN,
                "--out", RUN / "analysis"], "independent_analysis")
    run_logged([sys.executable, "-u", EXP / "diagnose.py", "--run", RUN,
                "--features", FEATURES], "mechanism_diagnostics")
    metadata["test_completed_utc"] = datetime.now(timezone.utc).isoformat()
    (RUN / "notebook_manifest.json").write_text(json.dumps(metadata, indent=2) + "\n")
else:
    print("Held-out stage skipped. Review and commit the freeze before enabling it.")

## Inspect completed evidence, then export

When the held-out stage has completed, the report distinguishes learned retrieval gains from the added predictive objective and from coordinate shortcuts. Lower future cosine loss alone is not sufficient evidence of identity preservation. Inspect the paired-seed comparison, absence/recovery, future identity diagnostics, independent audit and mechanism reliance diagnostics. The interventions can shift the checkpoint's input distribution; they are not retrained ablations.

The pinned `export_artifacts.py` first verifies all nine selected heads, 216 prediction archives, completed curves, analysis hashes, diagnostics and the complete 63-scene feature binding. Its deterministic `full_run.zip` preserves exact original run files, including learned weights and full logits/future arrays, plus dependency source snapshots. The separate `compact/` tree preserves reports, curves, plots, videos and value-exact scores/cells/eligibility, while omitting head weights and other prediction arrays. **The compact tree cannot rerun the full analysis or diagnostics standalone.** Commit compact evidence to Git; retain the full archive separately.

Both projected `cache/` tokens and the larger `full_features/` arrays are excluded from the standard archive; their manifest, projection and centering remain. They are reproducible but needed to rerun head inference and mechanism diagnostics. `INCLUDE_FEATURE_CACHES = False` keeps the optional, separate cache supplement disabled by default. Frozen backbone weights are excluded. Export stays skipped when no completed test report exists, and all notebook outputs remain empty in the repository.


In [ ]:
from IPython.display import display, Markdown, Video
INCLUDE_FEATURE_CACHES = False
EXPORT = Path("/content/day7_artifact_export")  # Use a fresh or empty destination.

if (RUN / "test" / "results.json").exists():
    display(Markdown((RUN / "test" / "RESULTS.md").read_text()))
    for report_path in sorted((RUN / "analysis").glob("*.md")):
        display(Markdown(report_path.read_text()))
    for video_path in sorted((RUN / "analysis").rglob("*.mp4")):
        display(Video(str(video_path), embed=True, width=768))
    diagnostic = json.loads((RUN / "diagnostics/mechanism_reliance_v1/reliance_results.json").read_text())
    print("Mechanism diagnostics completed for:", ", ".join(sorted(diagnostic["methods"])))
    # Run after diagnostics: the exporter deliberately refuses incomplete/stale evidence.
    subprocess.run([sys.executable, "-u", str(EXP / "export_artifacts.py"),
                    "--run", str(RUN), "--features", str(FEATURES), "--out", str(EXPORT)],
                   check=True, env=CHILD_ENV)
    exported = json.loads((EXPORT / "export_summary.json").read_text())
    print(json.dumps(exported, indent=2))
    compact_archive = Path(shutil.make_archive(str(EXPORT / "compact_evidence"), "zip",
                                             root_dir=EXPORT / "compact"))
    files.download(str(EXPORT / "full_run.zip"))
    files.download(str(compact_archive))
    files.download(str(EXPORT / "export_summary.json"))
    if INCLUDE_FEATURE_CACHES:
        cache_archive = Path("/content/day7_reproducible_feature_caches.zip")
        with zipfile.ZipFile(cache_archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
            for path in sorted(FEATURES.rglob("*")):
                if path.is_file():
                    bundle.write(path, "features/" + str(path.relative_to(FEATURES)))
        with zipfile.ZipFile(cache_archive) as bundle:
            assert bundle.testzip() is None
        print("Separate opt-in feature-cache supplement:", cache_archive)
        files.download(str(cache_archive))
else:
    print("Completed-test export skipped. Use the training review archive above until all held-out stages finish.")
